# H2b: Between-Bias Effect Comparison

This notebook tests whether the magnitude of the bias effect differs across the four bias types.

It keeps the same bias-specific effect definition used in H2a. All rows are fitted with one overall logistic regression in which each bias type $k$ has its own signed active-side dummy variable:

$$
\operatorname{logit}\left(P(y_i = 1)\right) = \alpha + \sum_{k} \beta_k X_{ik}
$$

where:

- $y_i = 1$ means participant $i$ selected Side A.
- $X_{ik} = +1$ when bias type $k$ is active on Side A.
- $X_{ik} = -1$ when bias type $k$ is active on Side B.
- $X_{ik} = 0$ otherwise, including every baseline row and every row that manipulates a different bias type.

All four bias types share the intercept $\alpha$, the baseline effect. Baseline rows have every $X_{ik} = 0$ (all $\beta_k$ terms drop out), so for them the model reduces to $\operatorname{logit}(P(y_i = 1)) = \alpha$: $\alpha$ is the log-odds of choosing Side A from the story alone, with no active bias on either side. The probability-scale effect for bias type $k$ is the same quantity reported in H2a:

$$
\Delta_k = \frac{1}{2}\left[\operatorname{logit}^{-1}(\alpha + \beta_k) - \operatorname{logit}^{-1}(\alpha - \beta_k)\right].
$$

H2b tests the omnibus null hypothesis that all four probability-scale effects are equal:

$$
H_0:
\Delta_{\mathrm{anchoring}}
=
\Delta_{\mathrm{fallacy}}
=
\Delta_{\text{pro-jargon}}
=
\Delta_{\mathrm{verbosity}}.
$$

The four point estimates come from the overall model fitted to the original data, exactly as in H2a. To estimate their joint sampling uncertainty, each bootstrap iteration resamples the full analysis dataframe once and refits the overall model on it, so all four effects in an iteration come from the same resampled data and the same refitted baseline effect.

The final omnibus test compares the three independent between-bias contrasts to their joint-bootstrap covariance matrix and reports a three-degree-of-freedom Wald chi-square test. Because the baseline effect $\alpha$ is shared, the same null can also be tested inside the model with a likelihood-ratio test against a common-coefficient model; that test is reported as a check.

## 1. Import packages

We use the same core packages as H2a. `statsmodels` fits the overall bias-specific logistic regression and the common-coefficient comparison model. `scipy` supplies the chi-square and normal-reference p-values. The bootstrap code reuses the same fast grouped logistic fit used in H2a.

In [1]:
from pathlib import Path
from itertools import combinations
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from scipy.special import expit, logit
from scipy.stats import chi2, norm
from statsmodels.stats.multitest import multipletests


## 2. Load the H2a results file

H2b uses the same final analysis file as H2a because it needs the shared baseline rows, the bias type for each manipulated row, the active side, and the selected side.

The notebook can be run either from the repository root or from the `results/` folder. The path check below supports both starting locations. `results.csv` is only read; it is not modified.


In [2]:
RESULTS_PATH = Path("results.csv")

if not RESULTS_PATH.exists():
    RESULTS_PATH = Path("results/results.csv")

# The participant's choice is stored as `adjudication`; the analysis calls it `selected_side`.
raw = pd.read_csv(RESULTS_PATH).rename(columns={"adjudication": "selected_side"})
raw.head()

,dialogue_id,dialogue_word_count,case,topic,bias_type,fallacy_subtype,variant_a,variant_b,ground_truth_side_order,dialogue_variant,selected_side,reward_type
0,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,A,base_plus_incentive
1,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,B,base_reward
2,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,B,base_reward
3,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,A,base_plus_incentive
4,1,717,our-quarterback-is-missing,Who was responsible for causing Eddie's disapp...,NaN,NaN,baseline,baseline,gt_first,primary-1,A,base_plus_incentive


## 3. Clean the analysis columns

The analysis uses the same four fields as H2a:

- `bias_type`: the bias family for manipulated rows. Baseline rows are `N/A`, which pandas reads as missing.
- `variant_a`: whether Side A was `active` or `baseline`.
- `variant_b`: whether Side B was `active` or `baseline`.
- `selected_side`: which side the participant selected.

This cell standardizes text values and turns blank `bias_type` values into the explicit label `baseline`.


In [3]:
df = raw.copy()
df.columns = df.columns.str.strip()

df["variant_a"] = df["variant_a"].astype(str).str.strip().str.lower()
df["variant_b"] = df["variant_b"].astype(str).str.strip().str.lower()
df["selected_side"] = df["selected_side"].astype(str).str.strip().str.upper()

bias_type = df["bias_type"].where(df["bias_type"].notna(), "baseline")
df["bias_type_clean"] = bias_type.astype(str).str.strip().str.lower()
df["bias_type_clean"] = df["bias_type_clean"].replace({"": "baseline", "nan": "baseline"})

df = df.loc[df["selected_side"].isin(["A", "B"])].copy()
df.shape


(369, 13)

## 4. Code the outcome and the bias-specific active-side dummies

This is the same coding used in H2a.

The outcome is `y`, where `1` means Side A was selected and `0` means Side B was selected.

`X` is the signed active-side indicator:

- `X = 0`: baseline row.
- `X = 1`: active bias appears on Side A.
- `X = -1`: active bias appears on Side B.

The overall model splits `X` into one signed dummy per bias type. `X_<bias_type>` equals `X` on that bias type's rows and `0` on every other row. A positive coefficient means choices move toward the side where that bias is active.

In [4]:
a_active = df["variant_a"].eq("active") & df["variant_b"].eq("baseline")
b_active = df["variant_a"].eq("baseline") & df["variant_b"].eq("active")
baseline = df["variant_a"].eq("baseline") & df["variant_b"].eq("baseline")

df["is_baseline"] = baseline
df["active_side"] = np.select(
    [a_active, b_active, baseline],
    ["A", "B", "baseline"],
    default="unexpected",
)

df["X"] = np.select(
    [df["active_side"].eq("A"), df["active_side"].eq("B")],
    [1, -1],
    default=0,
).astype(int)

df["y"] = df["selected_side"].eq("A").astype(int)

BIAS_LABELS = {
    "anchoring_bias": "anchoring",
    "fallacy_trait": "fallacy",
    "pro_jargon_bias": "pro-jargon",
    "verbosity_bias": "verbosity",
}

df["bias_label"] = df["bias_type_clean"].map(BIAS_LABELS).fillna(df["bias_type_clean"])

# One signed active-side dummy per bias type: X on that bias type's rows, 0 everywhere else.
BIAS_TERMS = {bias_type: f"X_{bias_type}" for bias_type in BIAS_LABELS}
for bias_type, term in BIAS_TERMS.items():
    df[term] = df["X"].where(df["bias_type_clean"].eq(bias_type), 0).astype(int)

df[[
    "bias_type_clean",
    "bias_label",
    "variant_a",
    "variant_b",
    "selected_side",
    "active_side",
    "X",
    *BIAS_TERMS.values(),
    "y",
]].head(12)

,bias_type_clean,bias_label,variant_a,variant_b,selected_side,active_side,X,X_anchoring_bias,X_fallacy_trait,X_pro_jargon_bias,X_verbosity_bias,y
0,baseline,baseline,baseline,baseline,A,baseline,0,0,0,0,0,1
1,baseline,baseline,baseline,baseline,B,baseline,0,0,0,0,0,0
2,baseline,baseline,baseline,baseline,B,baseline,0,0,0,0,0,0
3,baseline,baseline,baseline,baseline,A,baseline,0,0,0,0,0,1
4,baseline,baseline,baseline,baseline,A,baseline,0,0,0,0,0,1
5,baseline,baseline,baseline,baseline,A,baseline,0,0,0,0,0,1
6,baseline,baseline,baseline,baseline,A,baseline,0,0,0,0,0,1
7,baseline,baseline,baseline,baseline,A,baseline,0,0,0,0,0,1
8,baseline,baseline,baseline,baseline,B,baseline,0,0,0,0,0,0
9,baseline,baseline,baseline,baseline,A,baseline,0,0,0,0,0,1


## 5. Check that the coding matches the H2b design

H2b reuses the H2a design: baseline rows must have blank/baseline `bias_type`, and each manipulated row must have exactly one of the four bias labels. These checks stop the notebook if any row does not fit that structure.


In [5]:
assert not df["active_side"].eq("unexpected").any(), "Some rows do not match the expected active/baseline design."
assert df.loc[df["is_baseline"], "bias_type_clean"].eq("baseline").all(), "A baseline row has a non-baseline bias_type."
assert not df.loc[~df["is_baseline"], "bias_type_clean"].eq("baseline").any(), "A manipulated row is missing bias_type."

pd.crosstab([df["bias_label"], df["active_side"]], df["selected_side"], margins=True)


selected_side             A    B  All
bias_label active_side               
anchoring  A             24   14   38
           B             16   20   36
baseline   baseline      51   21   72
fallacy    A             25   11   36
           B             22   18   40
pro-jargon A             27   11   38
           B             21   16   37
verbosity  A             23   13   36
           B             15   21   36
All                     224  145  369

## 6. Identify the four bias types to compare

The order below is kept fixed throughout the notebook so that the model dummies, the original effect vector, the bootstrap draws, the covariance matrix, and the H2b contrasts all refer to the same four bias types.

In [6]:
BIAS_TYPES = [
    "anchoring_bias",
    "fallacy_trait",
    "pro_jargon_bias",
    "verbosity_bias",
]

BIAS_ORDER = [BIAS_LABELS[bias_type] for bias_type in BIAS_TYPES]
MODEL_TERMS = [BIAS_TERMS[bias_type] for bias_type in BIAS_TYPES]

bias_counts = (
    df.loc[~df["is_baseline"]]
    .groupby(["bias_type_clean", "bias_label", "active_side"])
    .size()
    .rename("n")
    .reset_index()
)

bias_counts

,bias_type_clean,bias_label,active_side,n
0,anchoring_bias,anchoring,A,38
1,anchoring_bias,anchoring,B,36
2,fallacy_trait,fallacy,A,36
3,fallacy_trait,fallacy,B,40
4,pro_jargon_bias,pro-jargon,A,38
5,pro_jargon_bias,pro-jargon,B,37
6,verbosity_bias,verbosity,A,36
7,verbosity_bias,verbosity,B,36


## 7. Define the same overall model used in H2a

This function:

1. Fits `logit(P(y = 1)) = alpha + sum_k beta_k X_k` to all rows.
2. Converts each bias type's coefficient into the same H2a probability-scale effect, using the shared baseline effect `alpha`.

The purpose here is to recover the four original H2a point estimates before doing any bootstrap resampling.

In [7]:
def probability_shift(alpha, beta):
    return 0.5 * (expit(alpha + beta) - expit(alpha - beta))


def fit_bias_specific_model(data):
    X_model = sm.add_constant(data[MODEL_TERMS], has_constant="add")
    model = sm.Logit(data["y"], X_model).fit(disp=False)

    effects = pd.Series(
        probability_shift(model.params["const"], model.params[MODEL_TERMS].to_numpy()),
        index=BIAS_ORDER,
    )

    return {
        "model": model,
        "effects": effects,
    }

## 8. Fit the original overall model

These are not bootstrap estimates. They are the four H2a point estimates from the overall model fitted to the original analysis dataframe. H2b compares these observed effects to the joint sampling uncertainty estimated below.

In [8]:
result = fit_bias_specific_model(df)

original_effects = pd.DataFrame(
    [
        {
            "bias_type": BIAS_LABELS[bias_type],
            "n_model": len(df),
            "n_baseline": int(df["is_baseline"].sum()),
            "n_active_side_a": int((df["bias_type_clean"].eq(bias_type) & df["active_side"].eq("A")).sum()),
            "n_active_side_b": int((df["bias_type_clean"].eq(bias_type) & df["active_side"].eq("B")).sum()),
            "beta_log_odds": result["model"].params[BIAS_TERMS[bias_type]],
            "effect": result["effects"][BIAS_LABELS[bias_type]],
            "effect_pp": 100 * result["effects"][BIAS_LABELS[bias_type]],
        }
        for bias_type in BIAS_TYPES
    ]
)

original_effects

,bias_type,n_model,n_baseline,n_active_side_a,n_active_side_b,beta_log_odds,effect,effect_pp
0,anchoring,369,72,38,36,0.389733,0.091727,9.172686
1,fallacy,369,72,36,40,0.302306,0.071454,7.145427
2,pro-jargon,369,72,38,37,0.304210,0.071899,7.189864
3,verbosity,369,72,36,36,0.474527,0.111111,11.111111


## 9. Define the fast H2a effect calculation for bootstrap samples

This is the same grouped logistic fit used in the H2a bootstrap code. It collapses rows into the experimental cells, fits the overall model with Newton-Raphson, and returns all four effects

$$
\Delta_k = \frac{1}{2}\left[\operatorname{logit}^{-1}(\alpha + \beta_k) - \operatorname{logit}^{-1}(\alpha - \beta_k)\right].
$$

Using this compact implementation makes it practical to refit the overall model in each of 10,000 joint bootstrap iterations.

In [9]:
def group_rows(y_values, design):
    # Rows with identical design values share a fitted probability, so they can be fitted as one binomial cell.
    cells, cell_index = np.unique(np.asarray(design, dtype=float), axis=0, return_inverse=True)
    cell_index = cell_index.ravel()
    totals = np.bincount(cell_index, minlength=len(cells)).astype(float)
    successes = np.bincount(cell_index, weights=np.asarray(y_values, dtype=float), minlength=len(cells))
    return successes, totals, cells


def grouped_logit_fit(successes, totals, design):
    overall_rate = successes.sum() / totals.sum()
    if overall_rate <= 0 or overall_rate >= 1:
        raise RuntimeError("Bootstrap sample has complete outcome separation.")

    params = np.zeros(design.shape[1])
    params[0] = logit(overall_rate)

    for _ in range(50):
        p = expit(design @ params)
        score = design.T @ (successes - totals * p)
        info = design.T @ (design * (totals * p * (1 - p))[:, None])

        step = np.linalg.solve(info, score)
        params += step

        if np.max(np.abs(step)) < 1e-10:
            break
    else:
        raise RuntimeError("Bootstrap model fit did not converge.")

    if not np.all(np.isfinite(params)):
        raise RuntimeError("Bootstrap model fit returned non-finite parameters.")

    return params


def bias_specific_effects(data):
    design = np.column_stack([np.ones(len(data)), data[MODEL_TERMS].to_numpy(dtype=float)])
    params = grouped_logit_fit(*group_rows(data["y"].to_numpy(), design))
    return probability_shift(params[0], params[1:])


# Confirm that the fast effect calculation exactly reproduces
# the four original H2a point estimates.
fast_effects = bias_specific_effects(df)
model_effects = result["effects"].loc[BIAS_ORDER].to_numpy()

if not np.allclose(fast_effects, model_effects, atol=5e-7):
    raise AssertionError(
        f"Fast effect calculation does not match the fitted model: {fast_effects} vs {model_effects}"
    )

print("All four fast effect calculations match the original fitted-model effects.")

All four fast effect calculations match the original fitted-model effects.


## 10. Joint bootstrap of the four H2a effects

H2b needs the **joint uncertainty of all four effects**. Each bootstrap iteration therefore:

1. Resamples the full final analysis dataframe with replacement, keeping the total sample size unchanged.
2. Refits the overall model once on that resampled dataframe.
3. Records the anchoring, fallacy, pro-jargon, and verbosity effects from that one fit.

This is the same resampling scheme as the H2a confidence intervals, but H2b uses the four draws jointly. Because all four effects in an iteration share the same refitted baseline effect `alpha`, the draws preserve the dependence through that shared baseline, which matters when comparing the four effects.

In [10]:
N_BOOTSTRAP = 10000
BOOTSTRAP_SEED = 20260903
BOOTSTRAP_WARN_THRESHOLD = 0.90


def joint_bootstrap_effects(
    data,
    n_boot=N_BOOTSTRAP,
    seed=BOOTSTRAP_SEED,
):
    rng = np.random.default_rng(seed)
    n = len(data)

    draws = []
    n_failed = 0

    for _ in range(n_boot):
        sample_positions = rng.integers(0, n, size=n)
        sample = data.iloc[sample_positions]

        try:
            draws.append(dict(zip(BIAS_ORDER, bias_specific_effects(sample))))

        except Exception:
            # The whole iteration is discarded if the overall model cannot be
            # estimated, because H2b needs a complete four-effect vector from
            # each bootstrap world.
            n_failed += 1

    draws = pd.DataFrame(draws, columns=BIAS_ORDER)

    if len(draws) == 0:
        raise RuntimeError("All joint bootstrap iterations failed.")

    if len(draws) < BOOTSTRAP_WARN_THRESHOLD * n_boot:
        warnings.warn(
            f"Only {len(draws)} of {n_boot} joint bootstrap iterations succeeded.",
            RuntimeWarning,
        )

    return {
        "draws": draws,
        "n_success": len(draws),
        "n_failed": n_failed,
    }


joint_bootstrap = joint_bootstrap_effects(df)
bootstrap_draws = joint_bootstrap["draws"]

print(
    f"Successful joint bootstrap iterations: "
    f"{joint_bootstrap['n_success']} / {N_BOOTSTRAP}"
)
bootstrap_draws.head(10)

Successful joint bootstrap iterations: 10000 / 10000


,anchoring,fallacy,pro-jargon,verbosity
0,0.087408,0.074361,-0.018487,0.138719
1,0.134946,0.028244,0.087482,-0.009588
2,0.120014,0.013895,0.138804,0.161784
3,0.067775,0.104167,0.139377,0.185830
4,0.163104,0.106697,0.085283,0.079892
5,0.118038,0.047837,0.009845,0.082957
6,0.164365,0.047435,0.013889,0.112746
7,0.132800,0.110288,0.039813,0.060425
8,0.056303,0.157604,-0.002363,0.030045
9,0.173361,0.090023,0.036982,0.148832


## 11. Joint-bootstrap standard errors and covariance

The original four effects remain the H2b point estimates.

The 10,000 bootstrap rows are used to estimate how those four point estimates vary together across repeated samples. The diagonal of the covariance matrix gives each effect's bootstrap variance; the off-diagonal entries capture how two effect estimates move together, including dependence induced by the shared baseline.

The correlation matrix is shown only as a diagnostic. The omnibus test uses the covariance matrix.


In [11]:
bootstrap_cov = bootstrap_draws.cov()
bootstrap_corr = bootstrap_draws.corr()
bootstrap_se = bootstrap_draws.std(ddof=1)

effect_uncertainty_table = original_effects[[
    "bias_type",
    "effect_pp",
]].copy()

effect_uncertainty_table["joint_bootstrap_se_pp"] = (
    effect_uncertainty_table["bias_type"]
    .map(bootstrap_se)
    * 100
)

print("Observed H2a effects with joint-bootstrap SEs:")
display(effect_uncertainty_table)

print("Joint-bootstrap covariance matrix (probability scale):")
display(bootstrap_cov)

print("Joint-bootstrap correlation matrix:")
display(bootstrap_corr)


Observed H2a effects with joint-bootstrap SEs:


,bias_type,effect_pp,joint_bootstrap_se_pp
0,anchoring,9.172686,5.810727
1,fallacy,7.145427,5.507328
2,pro-jargon,7.189864,5.526063
3,verbosity,11.111111,5.824308


Joint-bootstrap covariance matrix (probability scale):


,anchoring,fallacy,pro-jargon,verbosity
anchoring,0.003376,-0.000023,0.000005,0.000027
fallacy,-0.000023,0.003033,0.000040,0.000003
pro-jargon,0.000005,0.000040,0.003054,-0.000080
verbosity,0.000027,0.000003,-0.000080,0.003392


Joint-bootstrap correlation matrix:


,anchoring,fallacy,pro-jargon,verbosity
anchoring,1.000000,-0.007197,0.001553,0.007861
fallacy,-0.007197,1.000000,0.013138,0.000923
pro-jargon,0.001553,0.013138,1.000000,-0.024919
verbosity,0.007861,0.000923,-0.024919,1.000000


## 12. H2b omnibus test

H2b asks whether all four H2a effect magnitudes are equal:

$$
H_0: \Delta_A = \Delta_F = \Delta_P = \Delta_V.
$$

This is equivalent to requiring three independent differences to equal zero. Using anchoring as an arbitrary reference, define:

$$
D_1 = \Delta_A - \Delta_F,
$$

$$
D_2 = \Delta_A - \Delta_P,
$$

$$
D_3 = \Delta_A - \Delta_V.
$$

The choice of anchoring as the reference does not make anchoring special; any full-rank set of three contrasts tests the same equality null.

The observed contrast vector comes from the original H2a point estimates. Its covariance comes from the joint bootstrap. The omnibus Wald statistic is

$$
W = D^{\top} V_D^{-1}D,
$$

which is compared with a chi-square distribution with three degrees of freedom.


In [12]:
effect_vector = (
    original_effects
    .set_index("bias_type")
    .loc[BIAS_ORDER, "effect"]
    .to_numpy(dtype=float)
)

effect_cov = (
    bootstrap_cov
    .loc[BIAS_ORDER, BIAS_ORDER]
    .to_numpy(dtype=float)
)

# Anchoring is the reference only for constructing a full-rank
# set of three equality contrasts.
contrast_matrix = np.array(
    [
        [1.0, -1.0,  0.0,  0.0],  # anchoring - fallacy
        [1.0,  0.0, -1.0,  0.0],  # anchoring - pro-jargon
        [1.0,  0.0,  0.0, -1.0],  # anchoring - verbosity
    ]
)

observed_contrasts = contrast_matrix @ effect_vector
contrast_cov = contrast_matrix @ effect_cov @ contrast_matrix.T

wald_stat = float(
    observed_contrasts.T
    @ np.linalg.pinv(contrast_cov)
    @ observed_contrasts
)

h2b_df = int(np.linalg.matrix_rank(contrast_matrix))
h2b_p_value = float(chi2.sf(wald_stat, df=h2b_df))

contrast_table = pd.DataFrame(
    {
        "contrast": [
            "anchoring - fallacy",
            "anchoring - pro-jargon",
            "anchoring - verbosity",
        ],
        "observed_difference_pp": 100 * observed_contrasts,
        "bootstrap_se_pp": 100 * np.sqrt(np.diag(contrast_cov)),
    }
)

h2b_omnibus_table = pd.DataFrame(
    {
        "test": ["equal effect magnitude across four bias types"],
        "wald_chi2": [wald_stat],
        "df": [h2b_df],
        "p_value": [h2b_p_value],
    }
)

display(contrast_table)
h2b_omnibus_table


,contrast,observed_difference_pp,bootstrap_se_pp
0,anchoring - fallacy,2.027260,8.034665
1,anchoring - pro-jargon,1.982822,8.012629
2,anchoring - verbosity,-1.938425,8.194817


,test,wald_chi2,df,p_value
0,equal effect magnitude across four bias types,0.322709,3,0.955703


## Model-based likelihood-ratio check

All four effects come from one model with a shared baseline effect $\alpha$, and $\Delta_k$ increases strictly with $\beta_k$ when $\alpha$ is held fixed. The four probability-scale effects are therefore equal exactly when the four coefficients are equal.

That means the H2b null can also be tested inside the model: compare the overall bias-specific model with the H1 model, which has one common coefficient on `X` for all bias types. The likelihood-ratio test has three degrees of freedom and does not use the bootstrap. It is reported as a check on the bootstrap Wald test above, not as a replacement for it.

In [13]:
common_effect_model = sm.Logit(
    df["y"],
    sm.add_constant(df[["X"]], has_constant="add"),
).fit(disp=False)

h2b_lr_stat = float(2 * (result["model"].llf - common_effect_model.llf))
h2b_lr_df = len(MODEL_TERMS) - 1
h2b_lr_p_value = float(chi2.sf(h2b_lr_stat, df=h2b_lr_df))

h2b_lrt_table = pd.DataFrame(
    {
        "test": ["bias-specific model vs common-coefficient (H1) model"],
        "lr_statistic": [h2b_lr_stat],
        "df": [h2b_lr_df],
        "p_value": [h2b_lr_p_value],
    }
)

h2b_lrt_table

,test,lr_statistic,df,p_value
0,bias-specific model vs common-coefficient (H1)...,0.341762,3,0.951992


## Stratified joint bootstrap sensitivity analysis

The ordinary joint bootstrap above resamples 369 rows from the full analysis dataframe, so the number of observations in each experimental cell can vary across bootstrap datasets.

This sensitivity analysis keeps the original experimental allocation fixed. Each stratified bootstrap iteration resamples within the observed baseline cell and each observed bias-by-active-side cell, concatenates those nine resampled cells into one bootstrap dataframe, and refits the overall model on it. The same resampled baseline rows therefore inform the shared baseline effect $\alpha$ behind all four effects within that iteration.

In [14]:
STRATIFIED_BOOTSTRAP_SEED = 20260904


def experimental_cell_key(data):
    return np.where(
        data["is_baseline"],
        "baseline",
        data["bias_type_clean"] + "__" + data["active_side"],
    )


df = df.copy()
df["experimental_cell"] = experimental_cell_key(df)

EXPECTED_CELL_COUNTS = pd.Series(
    {
        "baseline": 72,
        "anchoring_bias__A": 38,
        "anchoring_bias__B": 36,
        "fallacy_trait__A": 36,
        "fallacy_trait__B": 40,
        "pro_jargon_bias__A": 38,
        "pro_jargon_bias__B": 37,
        "verbosity_bias__A": 36,
        "verbosity_bias__B": 36,
    },
    name="n",
)

observed_cell_counts = df["experimental_cell"].value_counts().reindex(
    EXPECTED_CELL_COUNTS.index
)
if not observed_cell_counts.equals(EXPECTED_CELL_COUNTS):
    raise AssertionError(
        "Observed experimental cell counts do not match the expected H2b design."
    )

if int(EXPECTED_CELL_COUNTS.sum()) != len(df):
    raise AssertionError("Expected cell counts do not sum to the analysis sample size.")

strata = {
    cell: df.loc[df["experimental_cell"].eq(cell)].copy()
    for cell in EXPECTED_CELL_COUNTS.index
}


def check_stratified_sample(sample):
    if len(sample) != len(df):
        raise AssertionError("Stratified bootstrap sample does not have N=369.")

    counts = sample["experimental_cell"].value_counts().reindex(
        EXPECTED_CELL_COUNTS.index,
        fill_value=0,
    )
    if not counts.equals(EXPECTED_CELL_COUNTS):
        raise AssertionError("Stratified bootstrap sample changed cell sizes.")


def stratified_joint_bootstrap_effects(
    n_boot=N_BOOTSTRAP,
    seed=STRATIFIED_BOOTSTRAP_SEED,
):
    rng = np.random.default_rng(seed)
    draws = []
    n_failed = 0

    for _ in range(n_boot):
        pieces = []
        for cell, n_cell in EXPECTED_CELL_COUNTS.items():
            stratum = strata[cell]
            sample_positions = rng.integers(0, len(stratum), size=int(n_cell))
            pieces.append(stratum.iloc[sample_positions])

        sample = pd.concat(pieces, ignore_index=True)
        check_stratified_sample(sample)

        try:
            draws.append(dict(zip(BIAS_ORDER, bias_specific_effects(sample))))
        except Exception:
            n_failed += 1

    draws = pd.DataFrame(draws, columns=BIAS_ORDER)
    if len(draws) == 0:
        raise RuntimeError("All stratified joint bootstrap iterations failed.")

    if len(draws) < BOOTSTRAP_WARN_THRESHOLD * n_boot:
        warnings.warn(
            f"Only {len(draws)} of {n_boot} stratified joint bootstrap iterations succeeded.",
            RuntimeWarning,
        )

    return {
        "draws": draws,
        "n_success": len(draws),
        "n_failed": n_failed,
    }


original_effect_check = result["effects"].loc[BIAS_ORDER].to_numpy(dtype=float)
if not np.allclose(original_effect_check, effect_vector, atol=5e-12):
    raise AssertionError("Original H2a point estimates changed before stratified bootstrap.")

stratified_joint_bootstrap = stratified_joint_bootstrap_effects()
stratified_bootstrap_draws = stratified_joint_bootstrap["draws"]

if stratified_bootstrap_draws.shape[1] != len(BIAS_TYPES):
    raise AssertionError("Stratified bootstrap did not return all four effects.")

print(
    f"Successful stratified joint bootstrap iterations: "
    f"{stratified_joint_bootstrap['n_success']} / {N_BOOTSTRAP}"
)
stratified_bootstrap_draws.head(10)


Successful stratified joint bootstrap iterations: 10000 / 10000


,anchoring,fallacy,pro-jargon,verbosity
0,0.145434,0.137902,0.098329,0.111111
1,0.118918,0.097479,0.125342,0.111111
2,0.145185,0.138385,0.138276,0.125000
3,0.077845,0.124465,0.058385,0.125000
4,0.011256,0.148735,0.072243,0.069444
5,0.145917,0.032131,0.058542,0.152778
6,0.024730,0.122629,0.045540,0.055556
7,0.105686,0.109960,0.112146,0.013889
8,0.064042,0.006988,-0.021767,0.097222
9,0.064642,0.018948,0.005198,0.041667


## Stratified-bootstrap standard errors, covariance, and H2b omnibus test

The observed point estimates and observed contrasts remain the original H2a estimates. Only the covariance matrix is re-estimated from the stratified joint bootstrap draws.


In [15]:
stratified_bootstrap_cov = stratified_bootstrap_draws.cov()
stratified_bootstrap_corr = stratified_bootstrap_draws.corr()
stratified_bootstrap_se = stratified_bootstrap_draws.std(ddof=1)

stratified_effect_uncertainty_table = original_effects[[
    "bias_type",
    "effect_pp",
]].copy()
stratified_effect_uncertainty_table["stratified_bootstrap_se_pp"] = (
    stratified_effect_uncertainty_table["bias_type"]
    .map(stratified_bootstrap_se)
    * 100
)

stratified_effect_cov = (
    stratified_bootstrap_cov
    .loc[BIAS_ORDER, BIAS_ORDER]
    .to_numpy(dtype=float)
)
stratified_contrast_cov = (
    contrast_matrix
    @ stratified_effect_cov
    @ contrast_matrix.T
)

stratified_wald_stat = float(
    observed_contrasts.T
    @ np.linalg.pinv(stratified_contrast_cov)
    @ observed_contrasts
)
stratified_h2b_df = h2b_df
stratified_h2b_p_value = float(chi2.sf(stratified_wald_stat, df=stratified_h2b_df))

stratified_contrast_table = contrast_table[[
    "contrast",
    "observed_difference_pp",
]].copy()
stratified_contrast_table["stratified_bootstrap_se_pp"] = (
    100 * np.sqrt(np.diag(stratified_contrast_cov))
)

stratified_h2b_omnibus_table = pd.DataFrame(
    {
        "test": ["equal effect magnitude across four bias types"],
        "wald_chi2": [stratified_wald_stat],
        "df": [stratified_h2b_df],
        "p_value": [stratified_h2b_p_value],
    }
)

print("Observed H2a effects with stratified joint-bootstrap SEs:")
display(stratified_effect_uncertainty_table)

print("Stratified joint-bootstrap covariance matrix (probability scale):")
display(stratified_bootstrap_cov)

print("Stratified joint-bootstrap correlation matrix:")
display(stratified_bootstrap_corr)

display(stratified_contrast_table)
stratified_h2b_omnibus_table


Observed H2a effects with stratified joint-bootstrap SEs:


,bias_type,effect_pp,stratified_bootstrap_se_pp
0,anchoring,9.172686,5.742100
1,fallacy,7.145427,5.468198
2,pro-jargon,7.189864,5.480099
3,verbosity,11.111111,5.663454


Stratified joint-bootstrap covariance matrix (probability scale):


,anchoring,fallacy,pro-jargon,verbosity
anchoring,0.003297,0.000014,0.000050,0.000040
fallacy,0.000014,0.002990,-0.000026,-0.000034
pro-jargon,0.000050,-0.000026,0.003003,0.000018
verbosity,0.000040,-0.000034,0.000018,0.003207


Stratified joint-bootstrap correlation matrix:


,anchoring,fallacy,pro-jargon,verbosity
anchoring,1.000000,0.004542,0.015936,0.012192
fallacy,0.004542,1.000000,-0.008690,-0.010832
pro-jargon,0.015936,-0.008690,1.000000,0.005792
verbosity,0.012192,-0.010832,0.005792,1.000000


,contrast,observed_difference_pp,stratified_bootstrap_se_pp
0,anchoring - fallacy,2.027260,7.911236
1,anchoring - pro-jargon,1.982822,7.874024
2,anchoring - verbosity,-1.938425,8.015824


,test,wald_chi2,df,p_value
0,equal effect magnitude across four bias types,0.343491,3,0.951652


## Ordinary versus stratified bootstrap comparison

The tables below compare the existing ordinary joint bootstrap with the stratified joint bootstrap sensitivity analysis. The point estimates are the same original H2a estimates in both cases.


In [16]:
effect_uncertainty_comparison = pd.DataFrame(
    {
        "Bias": BIAS_ORDER,
        "Observed effect (pp)": 100 * effect_vector,
        "Ordinary bootstrap SE (pp)": [100 * bootstrap_se[bias] for bias in BIAS_ORDER],
        "Stratified bootstrap SE (pp)": [100 * stratified_bootstrap_se[bias] for bias in BIAS_ORDER],
    }
)

contrast_uncertainty_comparison = pd.DataFrame(
    {
        "Contrast": contrast_table["contrast"],
        "Observed difference (pp)": contrast_table["observed_difference_pp"],
        "Ordinary bootstrap SE (pp)": contrast_table["bootstrap_se_pp"],
        "Stratified bootstrap SE (pp)": stratified_contrast_table["stratified_bootstrap_se_pp"],
    }
)

h2b_omnibus_comparison = pd.DataFrame(
    {
        "Bootstrap method": ["Ordinary joint bootstrap", "Stratified joint bootstrap"],
        "Wald chi-square": [wald_stat, stratified_wald_stat],
        "df": [h2b_df, stratified_h2b_df],
        "p": [h2b_p_value, stratified_h2b_p_value],
    }
)

print("Table 1: effect uncertainty")
display(effect_uncertainty_comparison)

print("Table 2: contrast uncertainty")
display(contrast_uncertainty_comparison)

print("Table 3: H2b omnibus")
h2b_omnibus_comparison


Table 1: effect uncertainty


,Bias,Observed effect (pp),Ordinary bootstrap SE (pp),Stratified bootstrap SE (pp)
0,anchoring,9.172686,5.810727,5.742100
1,fallacy,7.145427,5.507328,5.468198
2,pro-jargon,7.189864,5.526063,5.480099
3,verbosity,11.111111,5.824308,5.663454


Table 2: contrast uncertainty


,Contrast,Observed difference (pp),Ordinary bootstrap SE (pp),Stratified bootstrap SE (pp)
0,anchoring - fallacy,2.027260,8.034665,7.911236
1,anchoring - pro-jargon,1.982822,8.012629,7.874024
2,anchoring - verbosity,-1.938425,8.194817,8.015824


Table 3: H2b omnibus


,Bootstrap method,Wald chi-square,df,p
0,Ordinary joint bootstrap,0.322709,3,0.955703
1,Stratified joint bootstrap,0.343491,3,0.951652


## Optional stratified-bootstrap pairwise follow-up comparisons

These six pairwise comparisons are secondary to the omnibus H2b test. They use the stratified joint bootstrap draws to estimate each pairwise difference standard error and percentile interval, with Holm correction across the six raw p-values.


In [17]:
stratified_point_effect_map = original_effects.set_index("bias_type")["effect"].to_dict()
stratified_pairwise_rows = []

for bias_1, bias_2 in combinations(BIAS_ORDER, 2):
    observed_difference = stratified_point_effect_map[bias_1] - stratified_point_effect_map[bias_2]
    bootstrap_differences = (
        stratified_bootstrap_draws[bias_1]
        - stratified_bootstrap_draws[bias_2]
    )

    difference_se = bootstrap_differences.std(ddof=1)
    z_stat = observed_difference / difference_se
    raw_p = 2 * norm.sf(abs(z_stat))
    ci_low, ci_high = np.percentile(bootstrap_differences, [2.5, 97.5])

    stratified_pairwise_rows.append(
        {
            "comparison": f"{bias_1} - {bias_2}",
            "difference_pp": 100 * observed_difference,
            "stratified_bootstrap_se_pp": 100 * difference_se,
            "ci_low_pp": 100 * ci_low,
            "ci_high_pp": 100 * ci_high,
            "z": z_stat,
            "raw_p": raw_p,
        }
    )

stratified_pairwise_table = pd.DataFrame(stratified_pairwise_rows)
stratified_pairwise_table["holm_p"] = multipletests(
    stratified_pairwise_table["raw_p"],
    method="holm",
)[1]
stratified_pairwise_table["significant_holm_0_05"] = (
    stratified_pairwise_table["holm_p"] < 0.05
)

stratified_pairwise_table


,comparison,difference_pp,stratified_bootstrap_se_pp,ci_low_pp,ci_high_pp,z,raw_p,holm_p,significant_holm_0_05
0,anchoring - fallacy,2.027260,7.911236,-13.769830,17.667046,0.256251,0.797757,1.0,False
1,anchoring - pro-jargon,1.982822,7.874024,-12.952655,16.933455,0.251818,0.801182,1.0,False
2,anchoring - verbosity,-1.938425,8.015824,-17.112812,14.416167,-0.241825,0.808916,1.0,False
3,fallacy - pro-jargon,-0.044438,7.775184,-15.737176,15.097237,-0.005715,0.995440,1.0,False
4,fallacy - verbosity,-3.965685,7.914976,-19.121273,12.015921,-0.501036,0.616346,1.0,False
5,pro-jargon - verbosity,-3.921247,7.857907,-18.903093,11.334658,-0.499019,0.617766,1.0,False


## Stratified-bootstrap sensitivity interpretation

The ordinary joint bootstrap allows the numbers of observations in the experimental cells to vary across bootstrap resamples. The stratified joint bootstrap instead conditions on the observed experimental allocation and varies only which participants are sampled within each cell.

Comparing these two bootstrap schemes is a sensitivity analysis for whether the H2b conclusion depends on the resampling scheme. If the H2b omnibus result is non-significant under both methods, that supports the conclusion that there is no detectable evidence that bias effect magnitudes differ across bias types. This should not be interpreted as evidence that the four effects are exactly equal.


## 13. Optional pairwise follow-up comparisons

The omnibus H2b test answers whether there is evidence that **any** bias effect differs from the others. It does not identify which pair differs.

If the omnibus test is significant, the six pairwise comparisons below can be used as follow-up tests. Each pair uses the same joint-bootstrap draws to estimate the standard error of that effect difference. The six p-values are Holm-adjusted.

The table is computed regardless of the omnibus result so that it is available for inspection, but pairwise conclusions should be treated as follow-up analyses rather than as a replacement for the omnibus H2b test.


In [18]:
point_effect_map = (
    original_effects
    .set_index("bias_type")["effect"]
    .to_dict()
)

pairwise_rows = []

for bias_1, bias_2 in combinations(BIAS_ORDER, 2):
    observed_difference = (
        point_effect_map[bias_1]
        -
        point_effect_map[bias_2]
    )

    bootstrap_differences = (
        bootstrap_draws[bias_1]
        -
        bootstrap_draws[bias_2]
    )

    difference_se = bootstrap_differences.std(ddof=1)
    z_stat = observed_difference / difference_se
    raw_p = 2 * norm.sf(abs(z_stat))

    ci_low, ci_high = np.percentile(
        bootstrap_differences,
        [2.5, 97.5],
    )

    pairwise_rows.append(
        {
            "comparison": f"{bias_1} - {bias_2}",
            "difference_pp": 100 * observed_difference,
            "bootstrap_se_pp": 100 * difference_se,
            "ci_low_pp": 100 * ci_low,
            "ci_high_pp": 100 * ci_high,
            "z": z_stat,
            "raw_p": raw_p,
        }
    )

pairwise_table = pd.DataFrame(pairwise_rows)
pairwise_table["holm_p"] = multipletests(
    pairwise_table["raw_p"],
    method="holm",
)[1]
pairwise_table["significant_holm_0_05"] = (
    pairwise_table["holm_p"] < 0.05
)

pairwise_table


,comparison,difference_pp,bootstrap_se_pp,ci_low_pp,ci_high_pp,z,raw_p,holm_p,significant_holm_0_05
0,anchoring - fallacy,2.027260,8.034665,-13.590481,17.930403,0.252314,0.800798,1.0,False
1,anchoring - pro-jargon,1.982822,8.012629,-13.944163,17.456732,0.247462,0.804551,1.0,False
2,anchoring - verbosity,-1.938425,8.194817,-17.961739,13.945834,-0.236543,0.813012,1.0,False
3,fallacy - pro-jargon,-0.044438,7.750379,-15.085756,15.065025,-0.005734,0.995425,1.0,False
4,fallacy - verbosity,-3.965685,8.012116,-19.537613,12.040062,-0.494961,0.620628,1.0,False
5,pro-jargon - verbosity,-3.921247,8.127975,-19.579968,12.081469,-0.482438,0.629495,1.0,False


## 14. Publication-ready H2b tables

The first table shows the four original H2a point estimates together with the joint-bootstrap standard errors used for H2b.

The second table is the primary H2b result: a single three-degree-of-freedom omnibus test of whether the four effect magnitudes are equal. The model-based likelihood-ratio check is printed below it.

In [19]:
h2b_effect_table = pd.DataFrame(
    {
        "Bias": effect_uncertainty_table["bias_type"],
        "H2a effect (pp)": effect_uncertainty_table["effect_pp"].map(
            lambda value: f"{value:+.1f}"
        ),
        "Joint bootstrap SE (pp)": effect_uncertainty_table[
            "joint_bootstrap_se_pp"
        ].map(lambda value: f"{value:.1f}"),
    }
)

h2b_publication_table = pd.DataFrame(
    {
        "Test": ["Differences across bias types"],
        "Chi-square": [wald_stat],
        "df": [h2b_df],
        "p": [h2b_p_value],
    }
)

print("Bias-specific effects used in H2b:")
display(h2b_effect_table)

print(
    f"H2b omnibus: chi-square({h2b_df}) = {wald_stat:.3f}, "
    f"p = {h2b_p_value:.6f}"
)
print(
    f"Model-based check: LR chi-square({h2b_lr_df}) = {h2b_lr_stat:.3f}, "
    f"p = {h2b_lr_p_value:.6f}"
)

h2b_publication_table


Bias-specific effects used in H2b:


,Bias,H2a effect (pp),Joint bootstrap SE (pp)
0,anchoring,+9.2,5.8
1,fallacy,+7.1,5.5
2,pro-jargon,+7.2,5.5
3,verbosity,+11.1,5.8


H2b omnibus: chi-square(3) = 0.323, p = 0.955703
Model-based check: LR chi-square(3) = 0.342, p = 0.951992


,Test,Chi-square,df,p
0,Differences across bias types,0.322709,3,0.955703


## 15. Interpretation guide

The H2b point estimates are the same four probability-scale effects defined and estimated in H2a from the overall bias-specific model.

- The overall model fitted to the original data provides the four observed effects.
- The joint bootstrap does **not** replace those point estimates or average them into new effects.
- Each joint bootstrap iteration resamples the full analysis dataframe once and refits the overall model, so all four effects in an iteration come from the same fit.
- Because the four effects share the same refitted baseline effect $\alpha$ within each iteration, the bootstrap covariance matrix preserves their dependence through that shared baseline.
- The three-degree-of-freedom omnibus Wald test asks whether the observed between-bias differences are large relative to that joint sampling uncertainty.
- The model-based likelihood-ratio check tests the same null without the bootstrap, by comparing the overall model to a model with one common coefficient.
- A small H2b p-value provides evidence that at least one bias effect differs in magnitude from the others.
- A non-significant H2b p-value means there is not sufficient evidence that the four effect magnitudes differ; it does not establish that the four effects are identical.
- If the omnibus H2b test is significant, the Holm-adjusted pairwise table can be used to examine which bias pairs differ.

H2b is a hypothesis about the differences among the four H2a effects, not a new definition of the bias effect.